In [158]:
from utils import hamiltonian
from utils.structure import delete_overlaps
# from utils.plots import plot_with_center
# from typing import Tuple
import numpy as np
import matplotlib.pyplot as plt
from ase.visualize import view
import sisl
from sisl.physics import RecursiveSI
# import pandas as pd
from numba import njit
from scipy.spatial import cKDTree
# from scipy.sparse import csc_array

%load_ext line_profiler


The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


# Print matrix

In [153]:
def pretty_print_columns(A, decimals=2, zero_repr="0"):
    """
    Print a 2D NumPy array with per-column alignment.
    Handles complex numbers, negatives, and zeros gracefully.
    """
    A = np.asarray(A)
    if A.ndim != 2:
        raise ValueError("Input must be a 2D array")

    rows, cols = A.shape
    is_complex = np.iscomplexobj(A)

    # Build a string matrix (formatted values)
    str_matrix = np.empty(A.shape, dtype=object)
    for i in range(rows):
        for j in range(cols):
            val = A[i, j]
            if val == 0:
                s = zero_repr
            elif is_complex:
                s = f"{val.real:.{decimals}f}{val.imag:+.{decimals}f}j"
            else:
                s = f"{val:.{decimals}f}"
            str_matrix[i, j] = s

    # Compute per-column widths
    col_widths = [max(len(str_matrix[i, j]) for i in range(rows)) for j in range(cols)]

    # Print rows with proper per-column alignment
    for i in range(rows):
        row_str = " ".join(str_matrix[i, j].rjust(col_widths[j]) for j in range(cols))
        print(row_str)

# Make electrode and "arm"

In [ ]:
def get_coordinates(structure) -> np.ndarray:
    """Get atomic positions"""
    if isinstance(structure, sisl.Geometry):
        return structure.xyz
    elif isinstance(structure, np.ndarray):
        assert structure.shape[-1] == 3, "must be a 2D array with 3 columns (x, y, and z coords)"
        return structure
    else:
        raise TypeError("Input must be ndarray or sisl Geometry")

def _reorder_atoms(structure) -> sisl.Geometry:
    """order atoms with the propagation direction (x) as the last"""
    x, y, z = get_coordinates(structure).T
    order = np.lexsort((y,z,x))
    return structure.sub(order)


def build_electrode(width, length, kind="armchair") -> sisl.Geometry:
    vac = 3
    electrode = sisl.geom.graphene_nanoribbon(width, kind=kind, vacuum=vac)
    electrode = electrode.tile(length, axis=0)
    electrode = _reorder_atoms(electrode)
    return electrode

def build_nanoribbon(electrode: sisl.Geometry, center_size : int) -> sisl.Geometry:
    assert center_size > 0, "must have a center size of at least 1 `electrode`"
    return electrode.tile(2+center_size, 0)

WIDTH = 5
LENGTH = 2
electrode = build_electrode(WIDTH, LENGTH)
ribbon = build_nanoribbon(electrode, 1)

In [ ]:
def _mark_electrode(device: sisl.Geometry, electrode: sisl.Geometry):
    """Temporarily mark left/right electrodes for index finding"""
    N = len(electrode)
    device.atoms[:N] = sisl.Atom("N")
    device.atoms[-N:] = sisl.Atoms("O")
    
def _reset_atoms(device: sisl.Geometry): 
    """Reset all atoms to carbon"""
    device.atoms[:] = sisl.Atoms("C")
    

def find_electrode_indices(device: sisl.Geometry, num_electrodes: int) -> tuple[np.ndarray, np.ndarray]:
    """Return indices of left/right electrodes for each nanoribbon arm."""
    symbols = np.array([atom.symbol for atom in device.atoms])
    idx_N = np.where(symbols == "N")[0]
    idx_O = np.where(symbols == "O")[0]
    NN = len(idx_N) // num_electrodes
    NO = len(idx_O) // num_electrodes
    lefts, rights = [], []
    for i in range(num_electrodes):
        lefts.append(idx_N[i*NN:(i+1)*NN])
        rights.append(idx_O[i*NO:(i+1)*NO])
    return np.array(lefts), np.array(rights)


def find_nearest_atoms(coords: np.ndarray, point: np.ndarray, neighbours=6) -> np.ndarray:
    """Return indices of nearest atoms to point"""
    distances = np.linalg.norm(coords - point, axis=1)
    return np.argsort(distances)[:neighbours]

def guess_hexagon_center(structure):
    bond = 1.42
    distance_2_nn = 2*bond*np.cos(np.deg2rad(30))
    rtol = 5e-2
    coords = get_coordinates(structure)
    center = coords.mean(axis=0)
    atom_idx = find_nearest_atoms(coords, center, neighbours=6)
    hex_coords = coords[atom_idx]
    center = hex_coords.mean(axis=0)
    
    atom1, atom2 = hex_coords[[0,1]]
    if np.isclose(atom1[1], atom2[1], rtol=rtol) and np.isclose(atom1[1], center[1], rtol=rtol):
        print("Warning: The geometric center lies on bonds. Trying to shift the center by half atomic distance to 2. NN.")
        print(f"{'Atom 1':>20}:", atom1)
        print(f"{'Atom 2':>20}:", atom2)
        center += np.array([0, distance_2_nn/2, 0])  # shift the center by half the distance to the 2nd NN
        print(f"{'New Center':>20}:", center)
    return center
    
def find_overlap(structure: sisl.Geometry):
    coords = get_coordinates(structure)
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=0.1)
    return pairs

def build_reduced_device(nanoribbon: sisl.Geometry, electrode: sisl.geometry, *, repeat=3) -> tuple[sisl.Geometry, np.ndarray, np.ndarray]:
    """Build the reduced structure device along with indicis of left and right electrodes for each nanoribbon arm"""
    assert repeat in [1, 2, 3], "repeat can only be 1, 2, or 3"
    nanoribbon = nanoribbon.copy()
    
    origin_of_rotation = guess_hexagon_center(nanoribbon)
    rotation_axis = [0,0,1] # rotate around Z
    _mark_electrode(nanoribbon, electrode)
    
    device = nanoribbon.copy()
    for i in range(1, repeat):
        angle = 60
        angle = angle*(-1) if i%2 == 0 else angle
        rotated_nanoribbon = nanoribbon.rotate(angle=angle, v=rotation_axis, origin=origin_of_rotation)
        device += rotated_nanoribbon
    
    device = delete_overlaps(device)
    left_indices, right_indices = find_electrode_indices(device, repeat)
    _reset_atoms(device)
    device.set_nsc((1,1,1))
    
    return device, left_indices, right_indices

device, left_idx, right_idx = build_reduced_device(ribbon, electrode, repeat=3)

       Initial number of atoms: 180
 Atoms after removing overlaps: 144


array([1, 1, 1], dtype=int32)

In [185]:
atoms_style = []
for i, (l, r) in enumerate(zip(left_idx, right_idx)):
    atoms_style += [{"atoms": l, "size": i*0.3 + 0.5, "color": "blue"}]
    atoms_style += [{"atoms": r, "size": i*0.3 + 0.5, "color": "red"}]

device.plot(axes="xy", atoms_style=atoms_style)

In [ ]:
view(device.to.ase())

# Compute left/right self-energies for different k points and energies

In [ ]:
def _direction(Nk = 1, axis=1):
    "infer direction of extending kvector"
    assert type(axis) == int, "axis must be int"
    if not axis in range(3): # 0, 1, or 2
        raise ValueError("'axis' must be  0,  1,  or  2.")
    d = [1, 1, 1]
    d[axis] = Nk
    return d, Nk


def lr_energies(electrode: (sisl.Geometry | sisl.Hamiltonian | sisl.RecursiveSI), 
                En = 1e-5j, kvec = [0,0,0]) -> tuple[np.ndarray, np.ndarray]:
    if not isinstance(kvec, list):
        kvec = list(kvec)
    if isinstance(electrode, sisl.Geometry):
        # print("electrode is of type Geometry")
        H_0 = hamiltonian(electrode)
        SE = RecursiveSI(H_0, infinite="+A")
    elif isinstance(electrode, sisl.physics.RecursiveSI):
        # print("electrode is a self-energy object")
        SE = electrode
    elif isinstance(electrode, sisl.Hamiltonian):
        # print("electrode is of type Hamiltonian")
        SE = RecursiveSI(electrode, infinite="+A")
    else:
        raise ValueError("argument 'electrode' must be Geometry, Hamiltonian, or RecursiveSI")
    SE_L, SE_R = SE.self_energy_lr(E=En, k=kvec)
    return SE_L, SE_R

def add_lr_energies(Hk: np.ndarray,
                    lr_energies: tuple[np.ndarray, np.ndarray],
                    electrode_indices: tuple[np.ndarray, np.ndarray]
                    ) -> np.ndarray:
    
    SE_L, SE_R = lr_energies
    left_indices, right_indices = electrode_indices
    for arm, (lidx, ridx) in enumerate(zip(left_indices, right_indices)):
        Hk[np.ix_(lidx, lidx)] += SE_L
        Hk[np.ix_(ridx, ridx)] += SE_R
    return Hk  

H_E = hamiltonian(electrode)
SE = RecursiveSI(H_E, infinite="+A")
En = 1e-5j
kvec = [0,0,0]
SE_L, SE_R = lr_energies(SE, kvec=kvec, En=En)

H_D = hamiltonian(device)
Hk_with_lr_energies = H_D.Hk(k=kvec, format="array", dtype=complex)
# pretty_print_columns(Hk)

In [320]:
def inverse_no_jit(invMatrix):
    N = invMatrix.shape[0]
    I = np.eye(N)
    return np.linalg.solve(invMatrix, I)

@njit
def inverse_jit(invMatrix):
    return np.linalg.inv(invMatrix)
    

def compute_greens(Hk_with_lr_energies: np.ndarray, 
                   SkEn: np.ndarray, *, use_jit=False) -> np.ndarray:
    invG = SkEn - Hk_with_lr_energies
    if use_jit:
        return inverse_jit(invG)
    elif not use_jit:
        return inverse_no_jit(invG)
    else:
        raise ValueError(f"Optional argument 'use_jit' must be bool, but was {use_jit}")

def LDOS(G: np.ndarray) -> np.ndarray:
    return -(1/np.pi)*np.diag(G)

def main(device, lr_indices, electrode, E=1e-5j, kvec=(0,0,0)):
    H_D = hamiltonian(device)
    H_0 = hamiltonian(electrode)
    SE = RecursiveSI(H_0, infinite="+A")
    lrE = lr_energies(electrode=electrode, En=E, kvec=kvec)
    Hk = H_D.Hk(k=kvec, format="array", dtype=complex)
    Sk = H_D.Sk(k=kvec, format="array", dtype=complex)
    Hk = add_lr_energies(Hk, lrE, electrode_indices=lr_indices)
    G = compute_greens(Hk_with_lr_energies=Hk, SkEn=Sk*En, use_jit=True)
    return LDOS(G)

main(device=device, lr_indices=(left_idx, right_idx), electrode=electrode)


array([0.+1.58121150e-02j, 0.+1.58121150e-02j, 0.+4.31643744e-02j,
       0.+2.91092717e-07j, 0.+4.31643744e-02j, 0.+1.58120864e-02j,
       0.+1.19780689e-04j, 0.+1.58120864e-02j, 0.+4.31644031e-02j,
       0.+4.31644031e-02j, 0.+1.60508914e-02j, 0.+1.60508914e-02j,
       0.+4.64581316e-02j, 0.+3.29382006e-03j, 0.+4.64581316e-02j,
       0.+1.60508458e-02j, 0.+1.07573133e-03j, 0.+1.60508458e-02j,
       0.+1.41688664e-02j, 0.+1.41688664e-02j, 0.+1.82016691e-02j,
       0.+1.82016691e-02j, 0.+1.82016691e-02j, 0.+9.67906066e-03j,
       0.+1.82016691e-02j, 0.+9.67906066e-03j, 0.+9.67905783e-03j,
       0.+9.67906066e-03j, 0.+9.67905783e-03j, 0.+9.67905783e-03j,
       0.+9.67905783e-03j, 0.+9.67905783e-03j, 0.+9.67906066e-03j,
       0.+9.67905783e-03j, 0.+9.67906066e-03j, 0.+1.82016691e-02j,
       0.+9.67906066e-03j, 0.+1.82016691e-02j, 0.+1.82016691e-02j,
       0.+1.82016691e-02j, 0.+1.41688664e-02j, 0.+1.41688664e-02j,
       0.+1.60508458e-02j, 0.+1.07573133e-03j, 0.+1.60508458e-

In [323]:
def multi_LDOS(device: sisl.Geometry, electrode : sisl.Geometry, 
               lr_indices: tuple[np.ndarray, np.ndarray],
               *, energies: (np.ndarray | list) = [0], 
               Nk = 1, eta=1e-5) -> np.ndarray:
    
    k_direction, Nk = _direction(Nk=Nk, axis=1)
    Ne = len(energies)
    N_device = len(device)
    H_D = hamiltonian(device)
    kpts = sisl.MonkhorstPack(H_D, k_direction).k
    H_0 = hamiltonian(electrode)
    SE = RecursiveSI(H_0, infinite="+A")
    all_LDOS = np.zeros(shape=(Nk, Ne, N_device), dtype=complex)
    for ik, kvec in enumerate(kpts):
        Hk = H_D.Hk(k=kvec, format="array", dtype=complex)
        Sk = H_D.Sk(k=kvec, format="array", dtype=complex)
        for ie, E in enumerate(energies):
            En = E + 1j*eta
            SE_L, SE_R = lr_energies(electrode=SE, En=En, kvec=kvec)
            Hk = add_lr_energies(Hk, (SE_L, SE_R), lr_indices)
            G = compute_greens(Hk_with_lr_energies=Hk, SkEn=Sk*En, use_jit=True)
            all_LDOS[ik, ie, ...] = LDOS(G)
    return all_LDOS
multi_LDOS(device=device, electrode=electrode, lr_indices=(left_idx, right_idx))

array([[[0.+1.58121150e-02j, 0.+1.58121150e-02j, 0.+4.31643744e-02j,
         0.+2.91092717e-07j, 0.+4.31643744e-02j, 0.+1.58120864e-02j,
         0.+1.19780689e-04j, 0.+1.58120864e-02j, 0.+4.31644031e-02j,
         0.+4.31644031e-02j, 0.+1.60508914e-02j, 0.+1.60508914e-02j,
         0.+4.64581316e-02j, 0.+3.29382006e-03j, 0.+4.64581316e-02j,
         0.+1.60508458e-02j, 0.+1.07573133e-03j, 0.+1.60508458e-02j,
         0.+1.41688664e-02j, 0.+1.41688664e-02j, 0.+1.82016691e-02j,
         0.+1.82016691e-02j, 0.+1.82016691e-02j, 0.+9.67906066e-03j,
         0.+1.82016691e-02j, 0.+9.67906066e-03j, 0.+9.67905783e-03j,
         0.+9.67906066e-03j, 0.+9.67905783e-03j, 0.+9.67905783e-03j,
         0.+9.67905783e-03j, 0.+9.67905783e-03j, 0.+9.67906066e-03j,
         0.+9.67905783e-03j, 0.+9.67906066e-03j, 0.+1.82016691e-02j,
         0.+9.67906066e-03j, 0.+1.82016691e-02j, 0.+1.82016691e-02j,
         0.+1.82016691e-02j, 0.+1.41688664e-02j, 0.+1.41688664e-02j,
         0.+1.60508458e-02j, 0.+1.

In [ ]:
# H_D = hamiltonian(ribbon)
# H_0 = hamiltonian(electrode)
# L_energy, R_energy, _ = lr_energies(H_D, H_0)
# L_energy.shape, "identical shape :", R_energy.shape== L_energy.shape

((1, 1, 20, 20), 'identical shape :', True)

In [ ]:
# def device_hamiltonian(device, ribbon, electrode, left_idx, right_idx, **kwargs):
#     ribbon_HAM = hamiltonian(ribbon)
#     electrode_HAM = hamiltonian(electrode)
#     device_HAM = hamiltonian(device)
#     device_HAM.set_nsc((1,1,1)) # no PBC for structure
    
#     left_energies, right_energies, kwargs = lr_energies(ribbon_HAM, electrode_HAM, **kwargs)
#     assert left_energies.shape == right_energies.shape, "dimensions of left and right energies does not mathc..."
#     device = device.copy()
#     kpts = kwargs.get("kpts")
    
#     num_k, num_E = left_energies.shape[:2]
#     N_device = len(device)
    
#     hams = np.zeros(shape=(num_k, num_E, N_device, N_device), dtype=complex)
#     for iter_k, kvec in enumerate(kpts):
#         Hk = device_HAM.Hk(k=kvec, format="array").astype(complex)
#         for iter_E, E in enumerate(kwargs.get("energies")):
#             for iL, iR in zip(left_idx, right_idx):
#                 Hk[np.ix_(iL, iL)] += left_energies[iter_k, iter_E]
#                 Hk[np.ix_(iR, iR)] += right_energies[iter_k, iter_E]
#             hams[iter_k, iter_E, ...] = Hk
    
#     return hams

# device_hamiltonian(device, ribbon, electrode, left_idx, right_idx).shape
    

(1, 1, 144, 144)

In [324]:
# iL = left_idx[0]
# print(H_D.shape)
# H = H_D.Hk(format="array")
# H[np.ix_(iL, iL)]

In [ ]:
# @njit
# def hermconj(matrix):
#     "Hermitian conjugate of 2d matrix"
#     assert matrix.ndim == 2, "matrix must be 2D"
#     return matrix.T.conj()

# @njit
# def calc_gamma(se):
#     "Compute left/right broadening matrix from left/right self-energy"
#     return 1j*(se - hermconj(se))


# @njit
# def greens(left, right, E, H):
#     "compute freens function from Energy, device hamiltonian and left/right self-energies."
#     assert left.shape == right.shape, "left and right self-energies not identical."
#     N = len(left)
#     invG = E - H
#     invG[ :N,  :N] -= left
#     invG[-N:, -N:] -= right
#     return np.linalg.inv(invG)

# @njit
# def spectral(Greens, Gamma):
#     "compute left/right spectral function from greens function and left/right broadening matrix"
#     return Greens @ Gamma @ hermconj(Greens)


# def LDOS(device, electrode, energies, **kwargs):
#     T, A_ek, kpts = lr_energies(device, electrode, energies, **kwargs)
#     print(f"{A_ek.shape = }")
#     assert A_ek.shape[1] == len(energies), "number of energies and corresponding dimension of A does not match (check implementation)"
    
#     def rho(kidx, Eidx):
#         """Find LDOS from diagonal of spectral function/matrix"""
#         return np.diag(A_ek[kidx, Eidx]).real / (2*np.pi)
    
#     LDOS = np.zeros(shape=(*A_ek.shape[:2], A_ek.shape[-1]), dtype=float) # number of (k, E, A.shape) 
#     for iter_E, E in zip(range(A_ek.shape[1]), energies):
#         for iter_k in range(A_ek.shape[0]):
#             LDOS[iter_k, iter_E, :] = rho(iter_k, iter_E)
    
#     return T, LDOS, kpts

In [177]:
# Nk = 1
# NE = 50
# energies = np.linspace(-1, 1, num=NE)
# T, ldos, kpts = LDOS(H_D, H_0, energies); print(ldos.shape)

In [178]:
# import ipywidgets
# from ipywidgets import interact
# from fractions import Fraction

# def plotLDOS(ldos, kidx, site):
#     if isinstance(kidx, ipywidgets.Dropdown):
#         kidx = kidx.value
#     fig, ax = plt.subplots(1,1)
    
#     ymin, ymax = np.min(ldos), np.max(ldos)
    
#     ax.plot(energies, ldos[kidx, :, site])
#     ax.set_ylim(-ymax*0.1, ymax*1.01)
#     ax.set_xlabel("E")
#     ax.set_ylabel("LDOS")
    
#     # ax.legend()
# options = [(f"{[str(Fraction(val).limit_denominator(len(kpts)*2)) for val in kvec]})", i) for i, kvec in enumerate(kpts)]
# ks = ipywidgets.Dropdown(options=options, description="k")
# static = ipywidgets.fixed(ldos)
# sites = ipywidgets.IntSlider(min=0, max=ldos.shape[-1]-1, value=0, description="Site")

# interact(plotLDOS, ldos=static, kidx=ks, site=sites)
    

### Profiling

In [ ]:
def inverse1(G):
    return np.linalg.inv(G)
def inverse2(G):
    I = np.eye(G.shape[0])
    return np.linalg.solve(G, I)
def device_hamiltonian(G):
    inverse1(G)
    inverse2(G)    
    # return np.linalg.solve(G, I)
%lprun -f func func(G)

Timer unit: 1e-09 s

Total time: 0.0629423 s
File: /tmp/ipykernel_282758/1137498103.py
Function: func at line 6

Line #      Hits         Time  Per Hit   % Time  Line Contents
     6                                           def func(G):
     7         1   11338174.0 1.13e+07     18.0      inverse1(G)
     8         1   51604130.0 5.16e+07     82.0      inverse2(G)    
     9                                               # return np.linalg.solve(G, I)

In [179]:
# %lprun -f transport -s -u 1e-3 LDOS(PBC_ham, energies)